# Adatacore CoreForge — Chatbot LoRA (Research)
This notebook prepares an openly licensed dataset and fine-tunes a **Qwen3-0.6B LoRA adapter**. It does not replace your live Adatacore chatbot, deploy a production model, or consume paid GPUs by itself.

**Before running:** use Google Colab or Kaggle with GPU selected. Free GPU availability is not guaranteed. Review the Apache-2.0 Aya Hindi and Hinglish dataset cards and sample quality before training. Never add confidential customer or contributor data.\n

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before running training.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!git clone -q https://github.com/ajay333X/my-adatacore-site.git /content/my-adatacore-site || true
%cd /content/my-adatacore-site
!pip -q install 'unsloth' 'datasets>=3.0,<5' 'transformers>=4.51,<5' 'accelerate>=1.4,<2' 'peft>=0.15,<1'


## 1. Prepare public Hindi / Hinglish instruction data
This prepares human-annotated Aya Hindi and synthetic Hinglish examples with deterministic train/validation/test splits. The synthetic examples need quality review; preparing them does **not** train a model.\n

In [ ]:
!python workers/coreforge_hindi_data.py --sources aya_hindi,hinglish --limit 250 --accept-license --out /content/coreforge-hindi-data


## 2. Fine-tune Qwen3-0.6B with LoRA
This step performs real GPU training and evaluates on held-out examples. The output is a LoRA adapter, **not** an always-on chatbot server.


In [ ]:
!python workers/coreforge_chat_train.py --data /content/coreforge-hindi-data --output /content/coreforge-hindi-checkpoint --model Qwen/Qwen3-0.6B --max-steps 100 --approve-research-run


In [ ]:
import json
from pathlib import Path
report=json.loads(Path('/content/coreforge-hindi-checkpoint/training_report.json').read_text())
print(json.dumps(report,indent=2))


## 3. Save your adapter
Colab runtimes can disconnect. You must save the adapter directory to persistent storage (Drive, your own Hugging Face repository, or your own computer). It remains a **research checkpoint** until a real quality benchmark and inference deployment are completed.
